In [ ]:
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from statsmodels.tsa.stattools import coint
from itertools import combinations

# Atsisiunciame duomenis ( pvz . NVDA ir BTC )
tickers = ["BTC-USD", "DX-Y.NYB", "GC=F", "BZ=F"]
df = yf.download(tickers, start="2014-01-01")

# Suvienodinti dazni, nes portfeli sudaro kripto ir derivatyvai
df = df.dropna()

# Apskaiciuojame grazas
returns = df["Close"].pct_change().dropna()


[*********************100%***********************]  4 of 4 completed


In [ ]:
import pandas as pd

pd.set_option("display.max_rows", 500)
pd.set_option("display.max_columns", 500)
pd.set_option("display.width", 1000)


In [ ]:
# 1 užduotis
def compute_rsi(price_series: pd.Series, window: int = 14) -> pd.Series:
    delta = price_series.diff()

    gains = delta.clip(lower=0)
    losses = -delta.clip(upper=0)

    avg_gain = gains.rolling(window=window, min_periods=window).mean()
    avg_loss = losses.rolling(window=window, min_periods=window).mean()

    rs = avg_gain / avg_loss
    rsi = 100 - (100 / (1 + rs))
    return rsi


def compute_golden_cross_features(
    asset_df: pd.DataFrame, short_window: int = 50, long_window: int = 200
) -> pd.DataFrame:

    asset_df["ma50"] = (
        asset_df["Close"].rolling(window=short_window, min_periods=short_window).mean()
    )
    asset_df["ma200"] = (
        asset_df["Close"].rolling(window=long_window, min_periods=long_window).mean()
    )

    state = (asset_df["ma50"] > asset_df["ma200"]).fillna(False)
    prev_state = state.shift(1, fill_value=False)

    asset_df["in_golden_cross"] = state
    asset_df["golden_cross_event"] = state & (~prev_state)
    asset_df["death_cross_event"] = (~state) & prev_state

    return asset_df


def build_cross_features(
    asset_series: pd.Series, short_window: int = 50, long_window: int = 200
) -> pd.DataFrame:
    asset_df = pd.DataFrame({"Close": asset_series}).copy()

    asset_df = compute_golden_cross_features(asset_df, short_window, long_window)

    asset_df["rsi14"] = compute_rsi(asset_df["Close"], window=14)

    return asset_df


def analyze_assets(df: pd.DataFrame, weights: list[float]) -> dict[str, pd.DataFrame]:
    close_prices = df["Close"]

    assert np.isclose(sum(weights), 1), "Svoriai turi susideti i 1"
    assert (
        len(weights) == close_prices.shape[1]
    ), "Svoriu skaicius turi buti toks pats kaip ir aktyvu"

    assets_data: dict[str, pd.DataFrame] = {}
    for asset in close_prices.columns:
        asset_series = close_prices[asset]
        assets_data[asset] = build_cross_features(asset_series)

    return assets_data


In [ ]:
assets_data = analyze_assets(df, [0.5, 0.2, 0.1, 0.2])

btc = assets_data["BTC-USD"].copy()

# btc.loc[btc["golden_cross_event"] | btc["death_cross_event"], ["Close", "ma50", "ma200", "in_golden_cross", "golden_cross_event", "death_cross_event", "position"]]

,Close,ma50,ma200,in_golden_cross,golden_cross_event,death_cross_event,position
Date,,,,,,,
2015-11-03,403.416992,252.512740,250.332000,True,True,False,1
2018-05-18,8250.969727,8409.960752,8412.528285,False,False,True,0
2019-05-20,7978.309082,5247.699531,5187.610609,True,True,False,1
2019-12-16,6932.480469,8163.300684,8191.596292,False,False,True,0
2020-06-10,9870.094727,8463.689736,8449.205518,True,True,False,1
2021-07-23,33581.550781,36111.168359,36384.275903,False,False,True,0
2021-09-17,47267.519531,42390.074297,42353.810264,True,True,False,1
2022-02-02,36952.984375,46353.867734,46532.492949,False,False,True,0
2023-03-06,22429.757812,21120.036992,21107.803232,True,True,False,1


In [ ]:
btc[-20:]